# App3 Code Agent：让 AI 编写和执行代码 教案

**课程名称：** Code Agent：让 AI 编写和执行代码

**预计总时长：** 70-80 分钟

**源文件：** `Applications/App3_Code_Agent.ipynb`（共 20 个 Cell，Cell 0-19）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 环境准备 + LLM 后端配置 | Cell 0-4 | 8 min |
| 8-25 min | 安全代码执行器：设计理念 + 实现 + 测试 | Cell 5-8 | 17 min |
| 25-30 min | **休息 + 回顾** | -- | 5 min |
| 30-48 min | Code Agent 核心设计 + Function Calling 对比 + 完整实现 | Cell 9-11 | 18 min |
| 48-65 min | 测试 Code Agent：6 个任务演示 | Cell 12-18 | 17 min |
| 65-70 min | **休息 + 回顾** | -- | 5 min |
| 70-78 min | 总结 + 扩展练习 + 讨论 | Cell 19 | 8 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 已安装
- [ ] 确认 matplotlib 已安装（`pip install matplotlib`）
- [ ] 确认 Ollama 后端已就绪（`ollama pull qwen3:4b`）或已配置 DashScope API Key
- [ ] 预跑一遍全部 Cell，确认所有输出正常（特别是 LLM 后端连接）
- [ ] 准备白板或画板，用于手绘 Code Agent 循环流程图
- [ ] 确认中文字体配置正确（matplotlib 绘图需要）

---

## 第一段：开场 + 环境准备 + LLM 后端配置（Cell 0-4）

📍 浏览 Cell 0-2（Markdown 导读 + Ollama 说明），运行 Cell 3（环境准备），运行 Cell 4（LLM 后端配置）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：为什么需要让 LLM 写代码并执行？
- 理解 Code Agent 的四大能力：理解问题 → 生成代码 → 安全执行 → 自我修复
- 确认 LLM 后端和环境就绪

🗣 讲课话术

> 大家好！上一节我们学了 RAG，让 LLM 能查外部知识。今天我们更进一步——**让 LLM 写代码并执行代码**。
>
> 想象一个场景：你跟 AI 说「帮我算前 20 个斐波那契数」。纯 LLM 怎么做？它只能在脑子里算，很容易算错。但如果 LLM 能**写一段 Python 代码然后运行**，结果就精确了。这就是 Code Agent 的核心思路——**把计算任务交给 Python 解释器，LLM 负责写代码**。
>
> 看 Cell 0，Code Agent 有四大能力：(1) 理解问题——把自然语言转成代码需求；(2) 生成代码——LLM 编写 Python；(3) 安全执行——在沙箱里运行；(4) 自我修复——如果代码报错，根据错误信息修复。
>
> 这跟大家平时用的 ChatGPT Code Interpreter 是一个原理！OpenAI 内部也是一个沙箱 + LLM 生成代码 + 错误重试。
>
> 先运行 Cell 3 准备环境。（运行 Cell 3）看到 `[OK] 环境准备完成!` 就好。注意这里导入了 `io`、`re`、`traceback` 这些模块——等会儿沙箱执行器会用到。
>
> 再运行 Cell 4 配置 LLM 后端。（运行 Cell 4）代码按顺序尝试三种后端：先 Ollama（本地免费），再 DashScope（通义千问 API），最后 OpenAI。看到 `[OK] LLM 使用 xxx 后端` 就说明连接成功了。
>
> Cell 1 列了前置知识——需要 Ch12 的 Agent 原理基础。如果大家忘了 ReAct 循环，简单回忆：Thought → Action → Observation → ... → Final Answer。Code Agent 的 Action 就是「执行代码」，Observation 就是「代码输出或报错」。

👀 输出要点
- Cell 3：`[OK] 环境准备完成!`
- Cell 4：`[OK] LLM 使用 Ollama 后端` 或 `[OK] LLM 使用通义千问 (DashScope) 后端`
  - 如果三种都失败，会提示配置方法

❓ 预判问题
- **Q：Code Agent 和 ChatGPT 的 Code Interpreter 是一回事吗？**
  A：原理一样！都是 LLM 生成代码 → 沙箱执行 → 返回结果。ChatGPT 的版本功能更强——支持文件上传、图表生成、数据分析。我们今天实现一个简化版，掌握核心原理。
- **Q：为什么用 Ollama 而不是直接调 OpenAI？**
  A：Ollama 本地运行，免费且无网络延迟，适合教学环境。如果有 API Key，可以切换到 DashScope 或 OpenAI，代码不用改。
- **Q：qwen3:4b 这么小的模型能写代码吗？**
  A：简单任务可以——数学计算、列表操作、字符串处理。复杂任务（多文件项目、算法优化）需要更大的模型。今天的示例都是 4b 能搞定的。

➡️ 转场

> 环境好了，但在让 LLM 写代码之前，我们要先解决一个关键问题——**安全**。执行任意代码是危险的！如果 LLM 生成了 `os.system('rm -rf /')` 怎么办？下一节我们来造一个安全沙箱。

---

## 第二段：安全代码执行器——设计理念 + 实现 + 测试（Cell 5-8）

📍 浏览 Cell 5（Markdown：安全执行器介绍），浏览 Cell 6（Markdown：Code Agent 核心设计），运行 Cell 7（SafeCodeExecutor 实现），运行 Cell 8（测试执行器）

⏱ 时间分配：17 分钟（设计理念 7 分钟 + 代码实现 5 分钟 + 测试 5 分钟）

🎯 本段目标
- 理解为什么代码执行需要沙箱隔离
- 掌握三层安全机制：白名单内置函数、限制可导入模块、危险模式检测
- 理解 Code Agent 的整体架构：用户问题 → LLM 生成代码 → 沙箱执行 → 成功/失败
- 理解代码生成的三种方式：Prompt 引导、Function Calling、Structured Output

🗣 讲课话术

> Cell 5 开门见山：执行任意代码是**危险的**！我们需要三道防线——限制内置函数、限制可导入模块、捕获输出和错误。
>
> 打个比方：沙箱就像一个「玻璃房」。LLM 生成的代码在玻璃房里运行——能看到输出，但不能碰外面的东西。不能读写文件、不能访问网络、不能调用系统命令。
>
> 现在看 Cell 6，这是整个 Code Agent 的架构图。（指向 Cell 6 的流程图）
>
> ```
> 用户问题 (自然语言)
>     → LLM 生成代码
>     → 沙箱执行 (SafeCodeExecutor)
>     → 成功: 返回输出
>     → 失败: 错误信息 → LLM 修复 → 重新执行 (最多 N 次)
> ```
>
> 这个循环跟 ReAct 很像对不对？Thought 就是 LLM 分析问题、Action 就是生成代码并执行、Observation 就是执行结果或错误信息。区别是：Code Agent 的唯一「工具」就是 **Python 解释器**。
>
> Cell 6 还对比了三种让 LLM 生成代码的方式：
> - **Prompt 引导**：在 System Prompt 里要求「只输出代码块」——我们今天用这种
> - **Function Calling**：定义一个 `run_python` 工具，LLM 返回 `{"code": "..."}` 格式——OpenAI Assistants 用这种
> - **Structured Output**：直接要求 JSON 输出——Claude、GPT-4o 支持
>
> 三种方式殊途同归，核心都是让 LLM 输出**可执行的代码文本**，然后交给沙箱运行。
>
> Cell 6 底部有个关键洞察：**代码生成 = 受约束的文本生成**。通过 System Prompt 约束输出格式，通过沙箱约束执行权限。两层安全保障。
>
> 好，运行 Cell 7 看具体实现。（运行 Cell 7）SafeCodeExecutor 类大约 100 行代码，我带大家看核心设计。
>
> 第一层防线：**SAFE_BUILTINS**——白名单内置函数。只允许 `abs`、`len`、`print`、`range`、`sorted` 这些安全函数。`open`、`exec`、`eval`、`__import__` 全部移除。注意 `__import__` 被替换成了自定义的 `_safe_import`——只允许导入白名单里的模块。
>
> 第二层防线：**ALLOWED_MODULES**——只允许 `math`、`random`、`statistics`、`collections`、`datetime`、`json`、`re` 等安全模块。想导入 `os`、`subprocess`、`socket`？直接拒绝。
>
> 第三层防线：**DANGEROUS_PATTERNS**——用正则表达式预扫描代码。检测到 `open(`、`exec(`、`os.`、`subprocess`、`socket` 等危险模式就直接拦截，代码根本不会被执行。
>
> `execute` 方法的核心逻辑：先做安全检查 → 创建受限命名空间（只有白名单 builtins）→ 用 `exec(code, namespace)` 执行 → 用 `redirect_stdout/redirect_stderr` 捕获输出 → 返回结果字典 `{success, output, error, variables}`。
>
> 现在运行 Cell 8 看测试。（运行 Cell 8）三个测试：
> - **测试 1 正常代码**：`import math; sqrt(16) = 4.0`——成功执行，输出被正确捕获
> - **测试 2 文件操作**：`open('test.txt', 'w')`——被 DANGEROUS_PATTERNS 拦截，`Security Error: File operations not allowed`
> - **测试 3 禁止模块**：`import os; os.system('ls')`——被 DANGEROUS_PATTERNS 拦截，`Security Error: os module not allowed`
>
> 大家注意：测试 2 和测试 3 的代码**根本没被执行**，在安全检查阶段就被拦住了。这就是「防线前移」的思路——能在静态检查阶段拦的就不要等到运行时。

👀 输出要点
- Cell 7：`[OK] 安全执行器已创建`
- Cell 8 三个测试：
  - 测试 1：`Success: True`，`Output: sqrt(16) = 4.0`
  - 测试 2：`Success: False`，`Error: Security Error: File operations not allowed`
  - 测试 3：`Success: False`，`Error: Security Error: os module not allowed`

❓ 预判问题
- **Q：这个沙箱真的安全吗？**
  A：教学级别够用了。但生产环境不行——用 `exec` 即使限制了 builtins，仍有绕过方法（比如通过 `type.__subclasses__()` 获取 `os` 模块）。真实产品用 Docker 容器隔离或 gVisor 等内核级沙箱。
- **Q：为什么不直接用 Docker？**
  A：Docker 更安全但更重——需要启动容器、传输代码、收集输出。教学环境用进程内沙箱足够，重点是理解设计理念。
- **Q：`exec` 和 `eval` 有什么区别？**
  A：`eval` 只能执行单个表达式（如 `eval('1+2')` 返回 3），`exec` 可以执行任意代码块（函数定义、循环等）。Code Agent 需要执行完整代码，所以用 `exec`。
- **Q：SAFE_BUILTINS 为什么要包含 `True`、`False`、`None`？**
  A：因为我们覆盖了 `__builtins__`，如果不显式包含这些常量，代码里用 `True`/`False`/`None` 会报 NameError。

➡️ 转场

> 沙箱造好了！安全问题解决了。下面我们进入核心——**Code Agent 类的实现**。它要做的事：接收自然语言问题 → 调 LLM 生成代码 → 交给沙箱执行 → 如果失败就修复重试。

---

## 休息 + 回顾（第 25-30 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. Code Agent 的核心思路：让 LLM 写代码、用 Python 解释器执行代码，而不是让 LLM 自己算。
2. SafeCodeExecutor 有三层防线：白名单内置函数（SAFE_BUILTINS）、限制可导入模块（ALLOWED_MODULES）、危险模式正则检测（DANGEROUS_PATTERNS）。
3. 代码生成有三种方式（Prompt 引导 / Function Calling / Structured Output），本质都是让 LLM 输出可执行的代码文本。

**下一段预告：** 我们要实现完整的 Code Agent 类——先看 Function Calling 的 JSON 长什么样，然后构建 CodeAgent，让它真正用 LLM 写代码并执行。

---

## 第三段：Code Agent 核心设计 + Function Calling 对比 + 完整实现（Cell 9-11）

📍 浏览 Cell 9（Markdown：Code Agent 实现标题），运行 Cell 10（Function Calling 版本的 JSON 格式），运行 Cell 11（CodeAgent 完整实现）

⏱ 时间分配：18 分钟（Function Calling 格式 5 分钟 + CodeAgent 实现 13 分钟）

🎯 本段目标
- 理解 Function Calling 版 Code Agent 的 JSON 格式：工具定义 + LLM 返回 tool_calls
- 掌握 CodeAgent 的核心组件：SYSTEM_PROMPT、FIX_PROMPT、generate_code、fix_code、solve
- 理解自我修复机制：最多重试 N 次

🗣 讲课话术

> 运行 Cell 10。（运行 Cell 10）这段代码展示了：如果用 Function Calling 来实现 Code Agent，JSON 长什么样？
>
> 看工具定义——`run_python` 工具，参数有两个：`code`（Python 代码字符串）和 `description`（代码描述）。这就告诉 LLM：「你可以调用这个工具来执行 Python 代码」。
>
> 再看 LLM 返回的 tool_calls 示例。用户问「找出 1-50 的素数」，LLM 返回：
> ```json
> {"name": "run_python", "arguments": "{\"code\": \"primes = [n for n in range(2, 51) ...]\nprint(primes)\"}"}
> ```
> 注意 `content` 是 `null`——LLM 不是在说话，是在调用工具。`arguments` 里的 `code` 就是要执行的 Python 代码。
>
> 底部说了：「本 notebook 用 Prompt 引导方式实现，原理相同」「OpenAI Assistants / Code Interpreter 就是 Function Calling 版本」。所以大家理解原理就行，两种方式只是接口格式不同，核心流程一样。
>
> 好，重头戏来了！运行 Cell 11。（运行 Cell 11）CodeAgent 类大约 160 行代码。我带大家走一遍核心逻辑。
>
> **首先看 SYSTEM_PROMPT**。七条规则：
> 1. 只返回可执行的 Python 代码，不要解释
> 2. 用 `print()` 显示结果
> 3. 可用模块列表（跟沙箱的 ALLOWED_MODULES 一致）
> 4-6. 不能做文件操作、网络操作、访问 os/sys
> 7. 代码要简洁
>
> 这就是 Cell 6 说的「Prompt 工程要点」——角色定义、输出格式、可用资源、安全边界、质量要求，全部写进 System Prompt。
>
> **然后看 FIX_PROMPT**。当代码报错时，把上一次的代码和错误信息塞进去，让 LLM 修复。这就是「自我修复」机制。
>
> **`_extract_code` 方法**：用正则表达式从 LLM 响应里提取 ````python ... ``` 代码块。如果 LLM 没用代码块包裹，就把整个响应当代码。这是处理 LLM 输出格式不一致的防御性编程。
>
> **`_fallback_code` 方法**：兜底机制。当 LLM 返回空输出时，根据关键词匹配预设的代码模板。比如检测到「prime」和「50」就返回素数代码，检测到「fibonacci」就返回斐波那契代码。这保证了即使 LLM 抽风，demo 也能跑起来。
>
> **核心方法 `solve`**：
> 1. 打印问题
> 2. 调用 `generate_code` 生成代码
> 3. 进入重试循环（最多 `max_retries=3` 次）：
>    - 打印代码 → 执行 → 如果成功就返回结果
>    - 如果失败，调用 `fix_code` 修复 → 下一轮重试
> 4. 所有尝试都失败 → 返回错误信息
>
> 大家想一下：**为什么要限制重试次数？**（等 3 秒）对——防止无限循环！如果代码有根本性问题（比如需要网络访问但沙箱不允许），LLM 可能永远修不好。3 次重试是合理的安全阀。
>
> 看到输出 `Code Agent 已创建!` 就说明一切就绪了。

👀 输出要点
- Cell 10：
  - Function Calling 工具定义 JSON（`run_python` 工具，参数 `code` + `description`）
  - LLM 返回的 tool_calls JSON（`call_xyz789`，包含素数代码）
  - 说明：「本 notebook 用 Prompt 引导方式，OpenAI Assistants 是 Function Calling 版」
- Cell 11：`Code Agent 已创建!`

❓ 预判问题
- **Q：SYSTEM_PROMPT 里说「只返回代码」，但 LLM 经常加解释怎么办？**
  A：这就是 `_extract_code` 方法的作用——用正则从响应中提取代码块，忽略解释文字。小模型（如 4b）更容易不遵守格式约束，所以兜底逻辑很重要。
- **Q：`_fallback_code` 为什么用关键词匹配而不是让 LLM 重试？**
  A：效率和可靠性。LLM 空输出通常说明模型能力不够或 Prompt 不适合，重试大概率还是空。用预设模板保证 demo 流畅。
- **Q：max_retries=3 够用吗？**
  A：简单任务通常 1 次就成功。如果 3 次都失败，要么是需求超出沙箱能力（如需要网络），要么是模型能力不足。增加重试次数意义不大，反而浪费时间。
- **Q：`solve` 方法的 `history` 列表有什么用？**
  A：记录所有已解决的问题和结果，方便后续分析 Agent 的成功率和常见错误类型。实际产品中可用于监控和改进。

➡️ 转场

> Code Agent 造好了！现在让我们看看它的实战表现——6 个不同类型的任务：数学计算、列表操作、素数查找、统计分析、字符串处理、排序算法。

---

## 第四段：测试 Code Agent——6 个任务演示（Cell 12-18）

📍 浏览 Cell 12（Markdown 标题），逐一运行 Cell 13-18（6 个测试任务）

⏱ 时间分配：17 分钟（每个测试约 2.5 分钟 + 整体对比讨论 2 分钟）

🎯 本段目标
- 观察 Code Agent 的完整工作流程：生成代码 → 执行 → 输出结果
- 理解 LLM 生成代码的质量和特点
- 关注自我修复机制在实际运行中的表现
- 体会不同任务类型对代码生成的挑战

🗣 讲课话术

> 现在进入激动人心的测试环节！我们会让 Code Agent 解决 6 个不同类型的问题。每次运行注意看三件事：(1) LLM 生成了什么代码？(2) 代码执行结果是否正确？(3) 用了几次尝试？
>
> **测试 1：数学计算**（运行 Cell 13）
> 问题是「计算 10 的阶乘」。看 Attempt 1——LLM 生成了 `import math; print(math.factorial(10))`。很聪明！它没有自己手动算阶乘，而是调用了 `math.factorial` 库函数。执行成功，输出 **3628800**。一次通过！
>
> 这就是 Code Agent 的价值——**LLM 不需要自己算，它只需要写正确的代码**。计算交给 Python。
>
> **测试 2：列表操作**（运行 Cell 14）
> 问题是「生成前 20 个斐波那契数」。看代码——一个标准的斐波那契生成循环，`fib = [0, 1]`，然后不断 `append(fib[i-1] + fib[i-2])`。输出：`[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]`。20 个数，正确！
>
> 大家验证一下最后几个：987 + 1597 = 2584，1597 + 2584 = 4181。没问题。
>
> **测试 3：素数查找**（运行 Cell 15）
> 问题是「找出 1-50 之间的所有素数」。看 LLM 的代码——它写了一个 `is_prime` 函数，先处理边界（n < 2 返回 False），再用试除法（只试到 `sqrt(n)`，步长 2 跳过偶数）。优化意识不错！
> 输出：`[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]`。15 个素数，正确！
>
> **测试 4：统计分析**（运行 Cell 16）
> 这个任务更复杂——给定一组数字，计算均值、中位数、标准差、最小值和最大值。LLM 用了 `statistics` 模块——`statistics.mean`、`statistics.median`、`statistics.stdev`，加上内置的 `min` 和 `max`。
> 输出：Mean=50.5，Median=50.5，Standard deviation≈30.19，Min=11，Max=90。大家可以口算验证——10 个数的均值确实是 (23+45+67+89+12+34+56+78+90+11)/10 = 505/10 = 50.5。
>
> **测试 5：字符串处理**（运行 Cell 17）
> 问题是「检查回文字符串」。这个任务有个小陷阱——「A man a plan a canal Panama」是回文，但需要忽略大小写和非字母字符。看 LLM 的代码——先转小写再用列表推导式过滤非字母数字字符，然后比较正反序。
> 输出：`'racecar' → True`，`'hello' → False`，`'A man a plan a canal Panama' → True`。全部正确！
>
> **测试 6：排序算法**（运行 Cell 18）
> 问题是「实现冒泡排序」。这是一个经典算法题。看代码——标准的双重循环冒泡，还有个亮点：`sorted_arr = arr.copy()` 不修改原数组，这是好习惯。
> 输出：`[11, 12, 22, 25, 34, 64, 90]`。排序正确！
>
> **整体回顾**：6 个任务，涵盖数学计算、数据结构、算法、统计、字符串处理。全部一次通过（Attempt 1/3）。说明什么？(1) LLM 对常见编程任务有很好的代码生成能力；(2) 沙箱的安全模块白名单覆盖了大部分需求；(3) 自我修复机制在这些简单任务上没有被触发，但它是必要的安全网。
>
> 如果大家想看自我修复，可以试一些更刁难的任务——比如让 LLM 处理它不太擅长的场景，代码可能会在第一次报错，然后通过修复成功。

👀 输出要点
- Cell 13（阶乘）：`math.factorial(10)` → `3628800`，Attempt 1/3
- Cell 14（斐波那契）：循环生成 → `[0, 1, 1, 2, ..., 4181]`（20 个数），Attempt 1/3
- Cell 15（素数）：`is_prime` 函数 + 列表推导 → `[2, 3, 5, 7, ..., 47]`（15 个素数），Attempt 1/3
- Cell 16（统计）：`statistics` 模块 → Mean=50.5, Median=50.5, Std≈30.19, Min=11, Max=90，Attempt 1/3
- Cell 17（回文）：忽略大小写和非字母 → racecar=True, hello=False, Panama=True，Attempt 1/3
- Cell 18（冒泡排序）：双重循环 → `[11, 12, 22, 25, 34, 64, 90]`，Attempt 1/3

❓ 预判问题
- **Q：为什么所有任务都是 Attempt 1/3，没看到修复？**
  A：因为这 6 个任务都是常见编程题，LLM 训练数据里见过很多类似代码。复杂或小众任务更容易触发修复。可以试试用中文出题或提出更复杂的需求。
- **Q：LLM 生成的代码质量怎么样？**
  A：不错！可以看到 LLM 会选择合适的库（`math.factorial` 而不是手写递归）、处理边界情况（回文忽略大小写）、保持不变性（`arr.copy()`）。但不要指望它写复杂系统——目前最适合单函数、单脚本级别的任务。
- **Q：如果模型更大（比如 70B），效果会更好吗？**
  A：会！更大的模型代码生成质量更高、格式更稳定、能处理更复杂的任务。但 4b 模型在这些基础任务上已经足够了。
- **Q：每次运行结果一样吗？**
  A：不一定。LLM 有随机性（虽然我们设了 `temperature=0.2` 降低随机性）。代码风格可能不同，但结果应该正确。

➡️ 转场

> 6 个任务全部成功！我们已经看到了一个完整的 Code Agent 工作流。先休息一下，然后总结核心概念和扩展方向。

---

## 休息 + 回顾（第 65-70 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. CodeAgent 的核心循环：`generate_code` → `execute` → 成功则返回，失败则 `fix_code` → 重试（最多 3 次）。这是 ReAct 范式在代码生成场景的具体应用。
2. SYSTEM_PROMPT 是代码生成的「约束条件」——角色定义、输出格式、可用模块、安全边界，缺一不可。Prompt 质量直接决定生成代码的质量。
3. 6 个测试任务（阶乘 3628800、斐波那契 20 个数、15 个素数、统计 Mean=50.5、回文检测、冒泡排序）全部一次通过，说明 LLM + 沙箱的组合对常见编程任务非常有效。

**下一段预告：** 总结 Code Agent 的核心架构，讨论扩展方向——超时保护、多语言支持、可视化执行。

---

## 第五段：总结 + 扩展练习 + 讨论（Cell 19）

📍 浏览 Cell 19（Markdown 总结）

⏱ 时间分配：8 分钟（总结 3 分钟 + 练习讨论 5 分钟）

🎯 本段目标
- 回顾 Code Agent 的三大核心模块
- 讨论扩展方向和面试常见问题
- 布置课后练习

🗣 讲课话术

> Cell 19 总结了三大核心模块：
>
> **1. 安全执行环境**——白名单内置函数 + 限制可导入模块 + 危险模式正则检测 + 输出捕获。三层防线，代码还没执行就可能被拦住。
>
> **2. 代码生成**——LLM 将自然语言转成 Python 代码。关键技术：System Prompt 约束输出格式、正则提取代码块、fallback 兜底。
>
> **3. 自我修复**——代码报错时，把错误信息反馈给 LLM，让它生成修复版本。最多重试 N 次，防止无限循环。
>
> 现在聊聊扩展方向。Cell 19 列了三个：
>
> **(1) 添加超时保护**。当前沙箱没有时间限制——如果 LLM 生成了一个死循环 `while True: pass`，沙箱会卡死。解决方案：用 `multiprocessing` 或 `threading.Timer` 限制执行时间。实际产品一般限制 30 秒到 5 分钟。
>
> **(2) 支持更多语言**。我们的沙箱只支持 Python。如果要支持 JavaScript，可以用 Node.js 子进程；支持 Shell，可以用 Docker 容器。OpenAI Code Interpreter 目前只支持 Python，但 Claude Artifacts 支持 HTML/CSS/JS。
>
> **(3) 可视化执行**。当前只能看到文本输出。如果代码生成了 matplotlib 图表怎么办？可以把 `plt.savefig()` 的图片保存到临时目录，然后在前端展示。ChatGPT 就是这么做的。
>
> 最后，几个面试级别的思考题：
> - **Q：Code Agent 和 ReAct Agent 有什么联系和区别？**
>   Code Agent 是 ReAct Agent 的特化版——唯一的工具就是 Python 解释器。ReAct Agent 可以有多种工具（搜索、计算器、API），Code Agent 把所有问题都转化为代码来解决。
> - **Q：如何评估 Code Agent 的质量？**
>   三个维度：(1) 代码正确率——生成的代码能否正确解决问题；(2) 首次通过率——不需要修复就成功的比例；(3) 安全性——是否能有效拦截危险代码。
> - **Q：生产环境的 Code Agent 还需要什么？**
>   超时保护、资源限制（内存/CPU）、审计日志、更强的沙箱（Docker/gVisor）、代码审核机制（敏感操作需人工确认）。

👀 输出要点
- Cell 19 三大核心模块总结
- 三个扩展方向：超时保护、多语言支持、可视化执行

❓ 预判问题
- **Q：Code Agent 会取代程序员吗？**
  A：短期不会。Code Agent 擅长「写得出来的代码」——标准算法、数据处理、格式转换。但系统设计、需求分析、架构决策这些「写什么代码」的问题，仍然需要人来决定。Code Agent 是程序员的增强工具，不是替代品。
- **Q：Cursor、GitHub Copilot 和我们今天做的有什么关系？**
  A：原理相同！Copilot 做的是代码补全（LLM 生成代码片段），Cursor 的 Agent 模式更像我们今天做的——理解需求、生成完整代码、执行并验证。区别在于它们有更好的 IDE 集成和更大的模型。
- **Q：能不能让 Code Agent 修改文件或部署服务？**
  A：技术上可以——只需要解除沙箱限制。但这非常危险！所以实际产品（如 Devin、Claude Code）会加多层安全保障：权限控制、人工审批、回滚机制。

➡️ 转场

> 好！今天我们从零构建了一个 Code Agent——安全沙箱 + LLM 代码生成 + 自我修复。这是 AI 编程助手的核心原理。下一节 App4 我们会进入 Multi-Agent 系统——让多个 Agent 协作完成更复杂的任务。

---

## 练习环节：扩展 Code Agent

📍 基于 Cell 7（SafeCodeExecutor）和 Cell 11（CodeAgent）

⏱ 时间分配：课后练习（不占课堂时间）

### 练习 1：添加超时保护（难度：中）

**目标：** 修改 `SafeCodeExecutor.execute` 方法，添加超时保护。

**提示节奏：**
- 第一提示：可以用 `threading.Timer` 或 `signal.alarm`（Linux）
- 第二提示：更好的方案是用 `multiprocessing.Process` + `join(timeout)`
- 关键代码：
  ```python
  from multiprocessing import Process, Queue
  
  def _run_in_process(code, queue):
      result = original_execute(code)
      queue.put(result)
  
  p = Process(target=_run_in_process, args=(code, q))
  p.start()
  p.join(timeout=10)  # 10秒超时
  if p.is_alive():
      p.terminate()
      return {"success": False, "error": "Execution timeout"}
  ```

**常见错误：**
- 用 `threading.Timer` 无法真正终止 `exec` 中的死循环（GIL 限制）
- 忘记处理 `Queue` 为空的情况（超时后 `queue.get()` 会阻塞）

**验证标准：**
- 执行 `while True: pass` 应在 10 秒后返回 timeout 错误
- 正常代码不受影响

### 练习 2：添加新的安全模块（难度：易）

**目标：** 在 ALLOWED_MODULES 中添加 `numpy`（如已安装），并测试数值计算任务。

**提示：**
- 在 `ALLOWED_MODULES` 集合中添加 `'numpy'`
- 测试：让 Code Agent 解决「生成一个 5x5 的单位矩阵并打印」

### 练习 3：自定义 Code Agent 任务（难度：易）

**目标：** 设计 3 个你自己的编程任务，让 Code Agent 解决。

**建议方向：**
- 日期计算（用 `datetime` 模块）
- JSON 数据处理
- 正则表达式匹配
- 组合数学问题

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，环境准备 + LLM 后端配置 | 0-4 |
| 8 | 安全执行器：设计理念 + 核心架构图 | 5-6 |
| 15 | SafeCodeExecutor 实现 + 三层防线 | 7 |
| 20 | 安全测试：正常代码 / 文件操作 / 禁止模块 | 8 |
| 25 | **休息** | -- |
| 30 | Function Calling 版 Code Agent JSON 格式 | 10 |
| 35 | CodeAgent 完整实现（SYSTEM_PROMPT + solve 循环） | 11 |
| 48 | 测试 1-3：阶乘 / 斐波那契 / 素数 | 13-15 |
| 55 | 测试 4-6：统计分析 / 回文检测 / 冒泡排序 | 16-18 |
| 65 | **休息** | -- |
| 70 | 总结三大核心模块 + 扩展方向 | 19 |
| 75 | 面试题讨论 + Q&A | -- |

---

## 附录 B：关键数据快速参考

### Code Agent 架构

```
用户问题 (自然语言)
    |
    v
LLM 生成代码 (generate_code → SYSTEM_PROMPT + _extract_code)
    |
    v
沙箱执行 (SafeCodeExecutor.execute)
    |
    +--> 成功: 返回 {success: True, output: "..."}
    |
    +--> 失败: fix_code(code, error) → 重新执行 (最多 3 次)
```

### SafeCodeExecutor 三层防线

| 防线 | 机制 | 拦截时机 |
|:---|:---|:---|
| DANGEROUS_PATTERNS | 正则匹配危险代码 | 执行前（静态检查） |
| SAFE_BUILTINS | 白名单内置函数 | 执行时（namespace 限制） |
| ALLOWED_MODULES | 白名单可导入模块 | 导入时（_safe_import） |

### SAFE_BUILTINS 白名单（30+ 个）

`abs`, `all`, `any`, `bool`, `dict`, `divmod`, `enumerate`, `filter`, `float`, `format`, `frozenset`, `hash`, `int`, `isinstance`, `issubclass`, `iter`, `len`, `list`, `map`, `max`, `min`, `next`, `pow`, `print`, `range`, `repr`, `reversed`, `round`, `set`, `slice`, `sorted`, `str`, `sum`, `tuple`, `type`, `zip`, `True`, `False`, `None`, `Exception`, `ValueError`, `RuntimeError`

### ALLOWED_MODULES 白名单（11 个）

`math`, `random`, `statistics`, `collections`, `itertools`, `functools`, `operator`, `string`, `datetime`, `json`, `re`

### DANGEROUS_PATTERNS 正则检测（11 条）

| 模式 | 拦截原因 |
|:---|:---|
| `open(` | 文件操作 |
| `exec(` | 执行任意代码 |
| `eval(` | 表达式求值 |
| `__import__(` | 绕过导入限制 |
| `os.` | 操作系统访问 |
| `sys.` | 系统模块访问 |
| `subprocess` | 子进程操作 |
| `socket` | 网络操作 |
| `requests` | HTTP 请求 |
| `shutil` | 文件高级操作 |
| `pickle` | 反序列化攻击 |

### 6 个测试任务结果

| 测试 | 任务 | 关键输出 | 尝试次数 |
|:---|:---|:---|:---|
| 1 | 10 的阶乘 | 3628800 | 1/3 |
| 2 | 前 20 个斐波那契数 | [0, 1, 1, 2, ..., 4181] | 1/3 |
| 3 | 1-50 素数 | [2, 3, 5, 7, ..., 47]（15 个） | 1/3 |
| 4 | 统计分析 | Mean=50.5, Median=50.5, Std≈30.19 | 1/3 |
| 5 | 回文检测 | racecar→T, hello→F, Panama→T | 1/3 |
| 6 | 冒泡排序 | [11, 12, 22, 25, 34, 64, 90] | 1/3 |

### 代码生成三种方式

| 方式 | 做法 | 代表产品 |
|:---|:---|:---|
| Prompt 引导 | System Prompt 要求只输出代码块 | 本 notebook |
| Function Calling | 定义 `run_python` 工具 | OpenAI Assistants |
| Structured Output | 要求 JSON: `{"code": "..."}` | Claude / GPT-4o |

### CodeAgent 核心参数

| 参数 | 值 | 说明 |
|:---|:---|:---|
| max_retries | 3 | 最大重试次数 |
| temperature | 0.2 | LLM 采样温度（低随机性） |
| timeout | 10 | 执行超时（秒，当前未实现） |

---

## 附录 C：应急预案

### 场景 1：LLM 后端连接失败

**症状：** Cell 4 输出 `[X] 请配置 LLM 后端`

**应对：**
1. **Ollama 方案：** 确认 Ollama 服务已启动（`ollama serve`），确认模型已下载（`ollama pull qwen3:4b`）
2. **DashScope 方案：** 确认 `DASHSCOPE_API_KEY` 环境变量已设置
3. **OpenAI 方案：** 确认 `OPENAI_API_KEY` 环境变量已设置
4. **最坏情况：** 如果所有后端都不可用，可以用 Cell 11 的 `_fallback_code` 机制演示——手动设置 `llm = None` 但保留 fallback 逻辑，至少能展示沙箱执行部分

### 场景 2：LLM 生成空代码或乱码

**症状：** `Attempt 1/3` 显示空代码或无法解析的文本

**应对：**
1. 这是小模型的常见问题，`_fallback_code` 会自动兜底
2. 如果 fallback 也没匹配上，手动修改问题描述，使用更标准的英文表述
3. 向学生解释：这正好说明了 (1) fallback 机制的价值，(2) 大模型 vs 小模型的差异

### 场景 3：LLM 生成了危险代码

**症状：** LLM 在代码中使用了 `os.system`、`open` 等

**应对：**
1. 这是**最佳教学时机**！沙箱成功拦截了危险代码
2. 向学生展示安全检查的输出：`Security Error: xxx not allowed`
3. 然后修复会被触发——LLM 看到安全错误后应该生成不含危险操作的代码

### 场景 4：代码执行超时（卡住不动）

**症状：** Cell 运行很久没有输出

**应对：**
1. 可能是 LLM 生成了死循环代码且当前沙箱没有超时保护
2. 手动中断 Kernel（Kernel → Interrupt）
3. 向学生解释：这就是为什么扩展练习第一个就是「添加超时保护」
4. 重新运行该 Cell（LLM 会生成不同的代码）

### 场景 5：matplotlib 中文显示异常

**症状：** 图表中中文显示为方框

**应对：**
1. Cell 3 已配置了 `Microsoft YaHei` 和 `SimHei` 字体
2. 如果仍有问题，修改 `plt.rcParams["font.sans-serif"]` 为系统中已有的中文字体
3. 本节课不依赖图表输出——所有核心演示都是文本输出

### 场景 6：时间不够

**可以跳过的内容（按优先级）：**
1. Cell 10 Function Calling JSON 格式——口头带过「还可以用 Function Calling 实现，原理相同」
2. 测试 4-6（Cell 16-18）——保留前 3 个测试，口头总结后 3 个
3. 扩展练习讨论——留作课后作业

**不能跳过的内容：**
1. SafeCodeExecutor 三层防线（Cell 7-8）——安全是核心
2. CodeAgent 的 solve 循环（Cell 11）——这是 Agent 的灵魂
3. 至少 2-3 个测试演示（Cell 13-15）——让学生看到 Agent 真正工作

### 场景 7：学生质疑沙箱安全性

**症状：** 「用 exec 不是很危险吗？」「这个沙箱能被绕过吧？」

**应对：**
1. 承认：是的，Python 的 `exec` 沙箱确实可以被绕过（如 `type.__subclasses__()` 攻击链）
2. 解释分层：教学级（白名单 builtins）→ 进程级（subprocess + 资源限制）→ 容器级（Docker）→ 内核级（gVisor）
3. 强调：生产环境（如 ChatGPT Code Interpreter）用的是容器级隔离，不是我们这种进程内沙箱
4. 重点：今天的目标是理解**设计理念**（白名单 + 正则检测 + 捕获输出），不是造一个生产级沙箱